# Algorithmic shadow spectroscopy — reading energy gaps from time signals

We estimate an **energy gap** $\Delta E$ of a Hamiltonian from randomized (classical-shadow)
measurements of time-evolved states. The signal $\langle O(t)\rangle$ oscillates at the gap
frequencies; spectral analysis of many observables' signals reveals the gap as a peak.

We compare two ways to do the time evolution: deterministic **Trotter** (deep circuits) and
**TE-PAI** (shallow random circuits).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pai_shadow.hamil import Heisenberg_Hamil
from pai_shadow.trotter import trotter_circuit
from pai_shadow.shadow_spectro import (
    trotter_shadow_spectroscopy, te_pai_shadow_spectroscopy, dominant_gap)

np.random.seed(0)

In [ ]:
# 4-qubit Heisenberg chain
H = Heisenberg_Hamil(4, 1.0, 1.0, 1.0)

# initial state = ground + first excited  ->  signal oscillates at a single gap
E0, E1, ground, excited = H.get_ground_and_excited_state(n=1)
init = ground + excited
gap = E1 - E0
print(f"exact energy gap  Delta E = {gap:.4f}")

# time grid for the spectroscopy
Nt, dt = 50, 0.35
times = np.arange(Nt) * dt

## The time signal oscillates at the gap

For a state that is a superposition of two eigenstates, $\langle O(t)\rangle$ oscillates at the
transition frequency $\Delta E$. Here are a few **exact** expectation values vs time:

In [ ]:
plt.figure(figsize=(9, 4))
for obs in ["ZIII", "XXII", "IYYI"]:
    sig = [trotter_circuit(H, t, 100, init_state=init).expectation(obs) for t in times]
    plt.plot(times, sig, label=fr"$\langle {obs}\rangle$")
plt.xlabel("time $t$"); plt.ylabel("expectation")
plt.title(fr"Exact signals oscillate at $\Delta E={gap:.2f}$  (period ${2*np.pi/gap:.1f}$)")
plt.legend(); plt.tight_layout(); plt.show()

## Shadow spectroscopy

Instead of exact expectations, take **classical-shadow snapshots** at each time, estimate all
3-local Pauli observables, and spectrally analyse the resulting time x observable matrix. The
spectral peak sits at the gap. We run it with both Trotter and TE-PAI time evolution.

In [ ]:
ft, st = trotter_shadow_spectroscopy(H, init, times, n_steps=100, shadow_size=1200, k=3, seed=0)
fp, sp = te_pai_shadow_spectroscopy(H, init, times, delta=np.pi/8, M=1500, k=3, seed=0)
print(f"exact gap    = {gap:.3f}")
print(f"Trotter peak = {dominant_gap(ft, st):.3f}")
print(f"TE-PAI  peak = {dominant_gap(fp, sp):.3f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.axvline(gap, color="gray", ls="--", lw=1.5, label=f"exact gap {gap:.2f}")
plt.plot(ft, st / st.max(), color="tab:blue", label="Trotter shadow spectroscopy")
plt.plot(fp, sp / sp.max(), color="tab:red", label="TE-PAI shadow spectroscopy")
plt.xlim(0, 8)
plt.xlabel(r"frequency $\omega$  (= energy gap)"); plt.ylabel("intensity (normalised)")
plt.title("Recovered spectrum — peak at the energy gap")
plt.legend(); plt.tight_layout(); plt.show()

## Takeaways

- The spectral peak lands on the exact gap $\Delta E$ for **both** methods.
- TE-PAI uses much **shallower circuits** (its noise-robustness advantage), at the cost of quasiprobability sampling; its `n_steps` is chosen automatically so the angle stays near `delta`.
- Try a different excited index `n` in `get_ground_and_excited_state`, more observables (`k`), a larger `shadow_size`/`M`, or a longer `T` / smaller `dt` for sharper peaks.